In [5]:
import os
import subprocess

netlist_all = """\
Buck Converter CCM - All Variables
Vin in 0 DC 100
Vpwm gate 0 PULSE(0 5 0 1n 1n 0.000025 0.0001)
S1 in sw gate 0 SWMOD
D1 0 sw DMOD
L1 sw out 0.0006 IC=0
C1 out 0 50e-6 IC=0
RL out 0 2.5
.MODEL SWMOD SW(Ron=0.01 Roff=1MEG Vt=2.5)
.MODEL DMOD D(Is=1n N=1 Rs=0.01)
.TRAN 5e-7 0.002 UIC
.END
"""

netlist_save = """\
Buck Converter CCM - With SAVE
Vin in 0 DC 100
Vpwm gate 0 PULSE(0 5 0 1n 1n 0.000025 0.0001)
S1 in sw gate 0 SWMOD
D1 0 sw DMOD
L1 sw out 0.0006 IC=0
C1 out 0 50e-6 IC=0
RL out 0 2.5
.MODEL SWMOD SW(Ron=0.01 Roff=1MEG Vt=2.5)
.MODEL DMOD D(Is=1n N=1 Rs=0.01)
.TRAN 5e-7 0.002 UIC
.SAVE v(out) v(sw) i(l1) i(vin)
.END
"""

for name, netlist in [('all', netlist_all), ('save', netlist_save)]:
    cir = f'/tmp/buck_{name}.cir'
    raw = f'/tmp/buck_{name}.raw'
    with open(cir, 'w') as f:
        f.write(netlist)
    subprocess.run(['ngspice', '-b', '-r', raw, cir],
                   capture_output=True, text=True)
    size = os.path.getsize(raw)
    print(f'{name:6s} --> {size:,} bytes')

all    --> 321,902 bytes
save   --> 201,267 bytes


In [7]:
# binary 
netlist_bin = """\
RC Binary
Vs in 0 DC 5
R1 in vb 5000
C1 vb 0 100n IC=0
.TRAN 0.5u 2500u UIC
.END
"""

# ASCII
netlist_asc = """\
RC ASCII
Vs in 0 DC 5
R1 in vb 5000
C1 vb 0 100n IC=0
.TRAN 0.5u 2500u UIC
.OPTIONS FILETYPE=ASCII
.END
"""

for name, netlist in [('binary', netlist_bin), ('ascii', netlist_asc)]:
    cir = f'/tmp/rc_{name}.cir'
    raw = f'/tmp/rc_{name}.raw'
    with open(cir, 'w') as f:
        f.write(netlist)
    subprocess.run(['ngspice', '-b', '-r', raw, cir],
                   capture_output=True, text=True)
    size = os.path.getsize(raw)
    print(f'{name:8s} --> {size:,} bytes')



binary   --> 160,619 bytes
ascii    --> 490,234 bytes


In [8]:
# show what ASCII actually looks like
print('\nASCII raw file (first 20 lines)')
with open('/tmp/rc_ascii.raw') as f:
    for i, line in enumerate(f):
        if i >= 20: break
        print(line, end='')


ASCII raw file (first 20 lines)
Title: rc ascii
Date: Tue Jun  9 11:08:20  2026
Command: ngspice-46, Build Mon Mar 30 05:15:53 UTC 2026
Plotname: Transient Analysis
Flags: real
No. Variables: 4
No. Points: 5011    
Variables:
	0	time	time
	1	v(in)	voltage
	2	v(vb)	voltage
	3	i(vs)	current
Values:
0		5.000000000000000e-09
	5.000000000000000e+00
	4.999950000499996e-05
	-9.999900000999991e-04
1		1.000000000000000e-08
	5.000000000000000e+00
	9.999850001999976e-05


In [10]:
print('\nBinary raw file (first 20 lines)')
with open('/tmp/rc_binary.raw', 'rb') as f:
    data = f.read(400)

print(data)


Binary raw file (first 20 lines)
b'Title: rc binary\nDate: Tue Jun  9 11:08:20  2026\nCommand: ngspice-46, Build Mon Mar 30 05:15:53 UTC 2026\nPlotname: Transient Analysis\nFlags: real\nNo. Variables: 4\nNo. Points: 5011    \nVariables:\n\t0\ttime\ttime\n\t1\tv(in)\tvoltage\n\t2\tv(vb)\tvoltage\n\t3\ti(vs)\tcurrent\nBinary:\n:\x8c0\xe2\x8ey5>\x00\x00\x00\x00\x00\x00\x14@M\x9d\x1b\xbd\xd16\n?PB1\x16CbP\xbf:\x8c0\xe2\x8eyE>\x00\x00\x00\x00\x00\x00\x14@|\xeb &\xc96\x1a?\x0b\xe4wY8bP\xbf:\x8c0\xe2\x8eyU>\x00\x00\x00\x00\x00\x00\x14@\xe2\x81\xaeC\xbc6*?G:\x13\xe0"bP\xbf:\x8c0\xe2\x8eye>\x00\x00\x00\x00\x00\x00\x14@p\xe1\xb7\r\x9c6:?\xf1V\x9e\xed\xf7aP\xbf:\x8c0\xe2\x8e'
